# Sector-to-block/subloco assignment workflow

This notebook demonstrates a small end-to-end workflow for the sector-to-block/subloco assignment problem:

1. Generate synthetic historical orders for a set of sectors, each currently assigned to a (block, subloco) slot, plus the cycle calendar.
2. Forecast each sector's total cycle volume and its within-window order-day curve.
3. Reassign sectors as whole units to (block, subloco) slots with a capacity-aware, load-leveling heuristic.
4. Validate the forecast and compare as-is vs to-be daily captação (mean, standard deviation, peak).

#### Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## Problem setup

The company groups its ~800 sectors into 3 blocks and, within each block, 5 sublocos (closing days, Monday-Friday). A sector's (block, subloco) determines the start day of its 5-day sales window each cycle; its orders then spread across that window following a **fixed, sector-specific curve** (which day of the window its clients tend to buy on). Business rule: when a sector moves to a different block/subloco, that curve must be preserved -- only the window's calendar position changes.

Today, sectors are concentrated in block 2 (roughly matching the real ~45% concentration reported for this problem), which creates a capacity peak there while blocks 1 and 3 are underused. The assignment problem is to move whole sectors -- never split -- to different (block, subloco) slots so that daily order volume stays within capacity and is leveled across the cycle.

This is a teaching example: sectors, blocks and capacities are small and synthetic, and the heuristic is transparent rather than production-ready.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

SEED = 42
rng = np.random.default_rng(SEED)

BLOCKS = [1, 2, 3]
SUBLOCOS = [1, 2, 3, 4, 5]  # Monday .. Friday closing day within a block
SLOTS = [(block, subloco) for block in BLOCKS for subloco in SUBLOCOS]
WINDOW_LENGTH = 5  # days of sales window that follow a sector's assigned start day


def slot_start_day(block, subloco):
    """Day-in-cycle (1-indexed) on which a (block, subloco) slot's sales window opens."""
    return (block - 1) * len(SUBLOCOS) + subloco


CYCLE_LENGTH = len(SLOTS)  # 15: one start day per (block, subloco) slot
# Calendar days a cycle occupies, including window spillover past the last start day.
CYCLE_SPAN = CYCLE_LENGTH + WINDOW_LENGTH - 1
LAST_HISTORICAL_CYCLE = 6
VALIDATION_CYCLE = 7
N_CYCLES = VALIDATION_CYCLE

N_SECTORS = 24
SECTORS = [f"S{i:02d}" for i in range(1, N_SECTORS + 1)]


def build_current_assignment(rng, sectors, block_weights):
    """As-is (block, subloco) per sector, skewed toward block 2 (~45% real concentration)."""
    blocks = rng.choice(BLOCKS, size=len(sectors), p=[block_weights[b] for b in BLOCKS])
    sublocos = rng.choice(SUBLOCOS, size=len(sectors))
    return {
        sector: (int(block), int(subloco))
        for sector, block, subloco in zip(sectors, blocks, sublocos)
    }


def build_sector_weekday_shapes(rng, sectors):
    """Each sector's fixed within-window curve: share of orders per day offset in its window."""
    pattern_pool = rng.dirichlet(np.array([3, 4, 5, 4, 2]), size=len(sectors))
    return dict(zip(sectors, pattern_pool))


def build_sector_volume_parameters(rng, sectors):
    """Each sector's baseline order volume and its idiosyncratic scaling factor."""
    baseline = dict(zip(sectors, rng.uniform(20, 60, size=len(sectors))))
    factor = dict(zip(sectors, rng.uniform(0.85, 1.15, size=len(sectors))))
    return baseline, factor


# As-is calendar: skew most sectors into block 2, mirroring the real concentration
# problem (block 2 currently absorbs roughly 45% of order volume).
block_weights = {1: 0.15, 2: 0.65, 3: 0.20}
current_assignment = build_current_assignment(rng, SECTORS, block_weights)

# Each sector's fixed within-window curve: which day of its 5-day sales window its
# clients tend to order on. This shape must be preserved when a sector's block/subloco
# (and therefore its window's start day) changes.
sector_weekday_shape = build_sector_weekday_shapes(rng, SECTORS)

sector_baseline, sector_factor = build_sector_volume_parameters(rng, SECTORS)
cycle_factor = np.array([0.91, 0.96, 1.00, 1.04, 1.08, 1.12, 1.16])

## Part 1 -- Synthetic orders and cycle windows

Two core input tables:

- `orders`: one row per sector, cycle, and day-offset within its window -- the raw historical order feed, generated from each sector's current (block, subloco) assignment.
- `cycles`: one row per cycle, defining when the cycle opens and closes.

Each sector has a fixed within-window curve (`sector_weekday_shape`) describing how its orders split across the days of its 5-day window. This shape stays the same regardless of which slot the sector is assigned to -- only the calendar start day shifts.

In [ ]:
cycle_starts = pd.date_range("2025-01-01", periods=N_CYCLES, freq=f"{CYCLE_SPAN}D")
cycles = pd.DataFrame(
    {
        "cycle_id": np.arange(1, N_CYCLES + 1),
        "open_date": cycle_starts,
    }
)
cycles["close_date"] = cycles["open_date"] + pd.Timedelta(days=CYCLE_SPAN - 1)
cycles["n_days"] = (cycles["close_date"] - cycles["open_date"]).dt.days + 1

display(cycles)

In [ ]:
def expected_orders(sector, cycle_id, weekday_share):
    """Poisson mean orders for one sector-day, given its cycle position and window-day share."""
    return (
        sector_baseline[sector]
        * sector_factor[sector]
        * cycle_factor[cycle_id - 1]
        * weekday_share
        * WINDOW_LENGTH  # shape is a per-day share of the window; rescale to a volume
    )


def generate_sector_cycle_orders(rng, sector, cycle_id, cycle_start, assignment):
    """One sector's simulated order rows for a cycle, following its fixed within-window curve."""
    block, subloco = assignment[sector]
    start_day = slot_start_day(block, subloco)
    weekday_shape = sector_weekday_shape[sector]
    return [
        {
            "order_date": cycle_start + pd.Timedelta(days=start_day + offset - 1),
            "cycle_id": cycle_id,
            "day_in_cycle": start_day + offset,
            "weekday_offset": offset,
            "block": block,
            "subloco": subloco,
            "sector": sector,
            "orders": int(rng.poisson(expected_orders(sector, cycle_id, share))),
        }
        for offset, share in enumerate(weekday_shape)
    ]


def generate_orders(rng, sectors, cycle_starts, assignment):
    """Synthetic historical orders for every sector across all cycles, given an assignment."""
    rows = [
        row
        for cycle_id, cycle_start in enumerate(cycle_starts, start=1)
        for sector in sectors
        for row in generate_sector_cycle_orders(rng, sector, cycle_id, cycle_start, assignment)
    ]
    return pd.DataFrame(rows)


orders = generate_orders(rng, SECTORS, cycle_starts, current_assignment)
historical_orders = orders[orders["cycle_id"] <= LAST_HISTORICAL_CYCLE].copy()
validation_orders = orders[orders["cycle_id"] == VALIDATION_CYCLE].copy()

print(
    f"Generated {len(orders):,} sector-day observations across {len(SECTORS)} sectors "
    f"and {N_CYCLES} cycles."
)
print(
    f"Forecast training rows: {len(historical_orders):,}; "
    f"held-out validation rows: {len(validation_orders):,}."
)
display(orders.head(12))

## Part 2 -- Forecast the within-window curve and cycle volume

Same two-stage forecast as before, adapted to the sector/slot structure:

1. **Within-window shape:** for each sector, each historical day-offset's share of that sector's cycle total, averaged across historical cycles. This is keyed by day offset within the window (0-4), not by calendar day-in-cycle, since the calendar position depends on whichever slot the sector ends up in.
2. **Cycle volume:** a straight-line trend on each sector's historical cycle totals, extrapolated one cycle ahead.

The forecast for a sector/day-offset is `forecast cycle total x average day share`. This `forecast_weekday` table is the shared input for both the as-is calendar view below and the block/subloco reassignment in Part 3.

In [ ]:
historical_cycle_totals = (
    historical_orders.groupby(["cycle_id", "sector"], as_index=False)["orders"]
    .sum()
    .rename(columns={"orders": "cycle_total"})
)
shape_observations = historical_orders.merge(historical_cycle_totals, on=["cycle_id", "sector"])
shape_observations["order_share"] = shape_observations["orders"] / shape_observations["cycle_total"]
cycle_shape = shape_observations.groupby(["sector", "weekday_offset"], as_index=False)[
    "order_share"
].mean()


def extrapolate_cycle_total(cycle_totals):
    """Extrapolate one positive cycle total from a short linear trend."""
    values = cycle_totals.to_numpy(dtype=float)
    x_values = np.arange(len(values), dtype=float)
    slope = np.polyfit(x_values, values, deg=1)[0] if len(values) > 1 else 0.0
    return max(0.0, values[-1] + slope)


forecast_cycle_totals = (
    historical_cycle_totals.sort_values("cycle_id")
    .groupby("sector")["cycle_total"]
    .apply(extrapolate_cycle_total)
    .rename("forecast_cycle_total")
    .reset_index()
)

forecast_weekday = cycle_shape.merge(forecast_cycle_totals, on="sector")
forecast_weekday["forecast_orders"] = np.rint(
    forecast_weekday["order_share"] * forecast_weekday["forecast_cycle_total"]
).astype(int)

print("Forecast cycle totals by sector")
display(forecast_cycle_totals)
print("Average historical within-window shape (share of cycle volume per weekday offset)")
display(cycle_shape.pivot(index="weekday_offset", columns="sector", values="order_share"))

In [ ]:
target_cycle = cycles.loc[cycles["cycle_id"].eq(VALIDATION_CYCLE)].iloc[0]


def to_calendar(day_offsets_df, cycle_open_date, assignment):
    """Map (sector, weekday_offset) rows onto calendar dates for a given assignment."""
    frame = day_offsets_df.copy()
    starts = frame["sector"].map(lambda s: slot_start_day(*assignment[s]))
    frame["day_in_cycle"] = starts + frame["weekday_offset"]
    frame["order_date"] = cycle_open_date + pd.to_timedelta(frame["day_in_cycle"] - 1, unit="D")
    return frame


def plot_within_window_shapes(ax, cycle_shape, sectors):
    """Plot each sector's forecasted within-window order-share curve."""
    for sector in sectors:
        sector_shape = cycle_shape[cycle_shape["sector"].eq(sector)]
        ax.plot(
            sector_shape["weekday_offset"],
            sector_shape["order_share"],
            marker="o",
            label=f"Sector {sector}",
        )
    ax.set_title(f"Forecasted within-window shape (first {len(sectors)} sectors)")
    ax.set_xlabel("Day offset within the sector's window")
    ax.set_ylabel("Share of sector cycle volume")
    ax.legend(fontsize=8)


def plot_forecast_timeseries(ax, forecast_daily):
    """Plot the as-is calendar's forecast order time series for the validation cycle."""
    ax.plot(
        forecast_daily["order_date"],
        forecast_daily["forecast_orders"],
        marker="o",
        color="tab:blue",
        label="Forecast orders (as-is calendar)",
    )
    ax.set_title("Forecast time series for validation cycle (as-is assignment)")
    ax.set_xlabel("Order date")
    ax.set_ylabel("Orders")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()


forecast_as_is = to_calendar(forecast_weekday, target_cycle["open_date"], current_assignment)
forecast_daily = forecast_as_is.groupby("order_date", as_index=False)["forecast_orders"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# keep the legend readable with 24 sectors
plot_within_window_shapes(axes[0], cycle_shape, SECTORS[:8])
plot_forecast_timeseries(axes[1], forecast_daily)
fig.tight_layout()
plt.show()

### Alternative within-window shape-forecast strategies

The pipeline above (`cycle_shape`) forecasts each sector's within-window curve as a plain,
unweighted average of `order_share` across its 6 historical cycles. That is one choice among
several reasonable ones. This subsection formulates a few alternatives and scores them against
the held-out validation cycle, to make the tradeoff explicit rather than implicit:

- **Plain average** (current pipeline choice): equal weight per historical cycle. Simple, but
  ignores any drift in a sector's buying pattern over time.
- **Recency-weighted average (EWMA)**: exponentially decays older cycles' weight, so a sector's
  more recent behavior dominates the curve.
- **Median across cycles**: robust to a single noisy or anomalous cycle skewing the average.
- **Last-cycle-only**: uses only the most recent historical cycle, with no averaging at all --
  most reactive, but also the noisiest.
- **Shrinkage toward a global shape**: blends each sector's own average curve with the
  cross-sector average curve, which helps sectors whose own history is short or noisy.

The downstream pipeline (Part 3 onward) adopts whichever strategy scores best here -- see the
selection step after the level-forecast comparison below.

In [ ]:
def normalize_shape(shape_df):
    """Rescale each sector's weekday shares so they sum to 1."""
    totals = shape_df.groupby("sector")["order_share"].transform("sum")
    return shape_df.assign(order_share=shape_df["order_share"] / totals)


def shape_plain_average(shape_observations):
    """Within-window shape: unweighted average of order_share across historical cycles."""
    shape = shape_observations.groupby(["sector", "weekday_offset"], as_index=False)[
        "order_share"
    ].mean()
    return normalize_shape(shape)


def shape_recency_weighted(shape_observations, half_life_cycles=2.0):
    """Within-window shape: cycles weighted by recency via exponential decay (EWMA)."""
    frame = shape_observations.copy()
    cycles_ago = frame["cycle_id"].max() - frame["cycle_id"]
    frame["weight"] = 0.5 ** (cycles_ago / half_life_cycles)
    frame["weighted_share"] = frame["order_share"] * frame["weight"]
    grouped = frame.groupby(["sector", "weekday_offset"])
    shape = (
        (grouped["weighted_share"].sum() / grouped["weight"].sum())
        .rename("order_share")
        .reset_index()
    )
    return normalize_shape(shape)


def shape_median(shape_observations):
    """Within-window shape: median order_share across historical cycles, robust to a noisy cycle."""
    shape = shape_observations.groupby(["sector", "weekday_offset"], as_index=False)[
        "order_share"
    ].median()
    return normalize_shape(shape)


def shape_last_cycle(shape_observations):
    """Within-window shape: only the most recent historical cycle's share, with no averaging."""
    last_cycle_id = shape_observations["cycle_id"].max()
    columns = ["sector", "weekday_offset", "order_share"]
    shape = shape_observations.loc[shape_observations["cycle_id"].eq(last_cycle_id), columns]
    return normalize_shape(shape.reset_index(drop=True))


def shape_shrinkage(plain_average_shape, shrinkage=0.3):
    """Within-window shape: blend each sector's own average curve with the cross-sector curve."""
    global_shape = (
        plain_average_shape.groupby("weekday_offset", as_index=False)["order_share"]
        .mean()
        .rename(columns={"order_share": "global_share"})
    )
    blended = plain_average_shape.merge(global_shape, on="weekday_offset")
    own_share = (1 - shrinkage) * blended["order_share"]
    global_contribution = shrinkage * blended["global_share"]
    blended["order_share"] = own_share + global_contribution
    return normalize_shape(blended[["sector", "weekday_offset", "order_share"]])


plain_average_shape = shape_plain_average(shape_observations)
shape_strategies = {
    "plain_average": plain_average_shape,
    "recency_weighted": shape_recency_weighted(shape_observations),
    "median": shape_median(shape_observations),
    "last_cycle": shape_last_cycle(shape_observations),
    "shrinkage": shape_shrinkage(plain_average_shape),
}

In [ ]:
def forecast_with_shape(shape_df, forecast_cycle_totals):
    """Combine a within-window shape forecast with the cycle-total forecast into daily counts."""
    forecast = shape_df.merge(forecast_cycle_totals, on="sector")
    forecast["forecast_orders"] = np.rint(
        forecast["order_share"] * forecast["forecast_cycle_total"]
    ).astype(int)
    return forecast


def score_shape_strategy(
    shape_df, forecast_cycle_totals, assignment, cycle_open_date, actual_daily
):
    """Forecast daily orders for one shape strategy and score it against the actual cycle."""
    forecast = forecast_with_shape(shape_df, forecast_cycle_totals)
    forecast = to_calendar(forecast, cycle_open_date, assignment)
    strategy_daily = forecast.groupby("order_date", as_index=False)["forecast_orders"].sum()
    comparison = strategy_daily.merge(actual_daily, how="outer", on="order_date").fillna(0)
    error = comparison["forecast_orders"] - comparison["actual_orders"]
    mae = error.abs().mean()
    wmape = error.abs().sum() / comparison["actual_orders"].sum()
    return strategy_daily, mae, wmape


actual_daily = (
    validation_orders.groupby("order_date", as_index=False)["orders"]
    .sum()
    .rename(columns={"orders": "actual_orders"})
)

strategy_scores = []
strategy_daily_series = {}
for name, shape_df in shape_strategies.items():
    strategy_daily, mae, wmape = score_shape_strategy(
        shape_df, forecast_cycle_totals, current_assignment, target_cycle["open_date"], actual_daily
    )
    strategy_daily_series[name] = strategy_daily
    strategy_scores.append({"strategy": name, "daily_MAE": mae, "daily_WMAPE": wmape})

strategy_scoreboard = pd.DataFrame(strategy_scores).set_index("strategy").sort_values("daily_WMAPE")
print("Shape-forecast strategies scored on the held-out validation cycle (as-is calendar)")
display(strategy_scoreboard)

In [ ]:
def plot_shape_strategies(ax, shape_strategies, sector):
    """Plot each strategy's within-window shape curve for one sector."""
    for name, shape_df in shape_strategies.items():
        sector_shape = shape_df[shape_df["sector"].eq(sector)].sort_values("weekday_offset")
        ax.plot(sector_shape["weekday_offset"], sector_shape["order_share"], marker="o", label=name)
    ax.set_title(f"Sector {sector}: shape-forecast strategies")
    ax.set_xlabel("Day offset within window")
    ax.set_ylabel("Share of cycle volume")
    ax.legend(fontsize=8)


def plot_strategy_scores(ax, strategy_scoreboard):
    """Plot each strategy's WMAPE against the held-out validation cycle."""
    strategy_scoreboard["daily_WMAPE"].plot(kind="bar", ax=ax, color="tab:blue")
    ax.set_title("Strategy accuracy on the held-out cycle")
    ax.set_ylabel("Daily WMAPE")
    ax.tick_params(axis="x", rotation=20)


# illustrative examples; every sector's curves are in shape_strategies
compare_sectors = SECTORS[:2]
fig, axes = plt.subplots(1, len(compare_sectors) + 1, figsize=(6 * (len(compare_sectors) + 1), 5))
for ax, sector in zip(axes, compare_sectors):
    plot_shape_strategies(ax, shape_strategies, sector)
plot_strategy_scores(axes[-1], strategy_scoreboard)
fig.tight_layout()
plt.show()

### Alternative cycle-total (level) forecast strategies

The pipeline above (`extrapolate_cycle_total`, inside `forecast_cycle_totals`) forecasts each
sector's next cycle total with a degree-1 linear trend fit on its 6 historical cycle totals.
As with the shape curve, that is one choice among several. This subsection formulates a few
alternatives and scores them against each sector's actual cycle total in the held-out cycle:

- **Naive last-value**: forecasts next cycle = last observed cycle. The simplest possible
  baseline; useful as a sanity floor for the others.
- **Linear trend** (current pipeline choice): extrapolates a straight-line fit one step ahead.
- **ETS / Holt's linear trend**: exponential smoothing with a trend component (`statsmodels`'
  `Holt`) -- similar spirit to the linear trend, but weights recent cycles more heavily.
- **ARIMA(1,1,0)**: a small autoregressive-integrated model (`statsmodels`' `ARIMA`) on the
  differenced series.
- **Regression with calendar regressors**: OLS on cycle index + the cycle's calendar month,
  in case volume has a seasonal calendar component beyond a pure trend.

**Note on scope:** with only 6 historical points per sector, this is a genuinely short series.
ARIMA/ETS are already at the edge of what's meaningfully fittable here, and results should be
read as illustrative, not as evidence one method is truly better. Prophet is left out entirely:
it targets much longer series with real seasonality/holiday structure (which this synthetic
data doesn't have -- the within-window shape curve already captures the only cyclical pattern),
and pulls in a heavy dependency (`cmdstanpy`) that isn't worth it for a 6-point series.

Requires the `forecast` extra (`uv sync --extra forecast`) for `statsmodels`. The downstream
pipeline (Part 3 onward) adopts whichever strategy scores best here, alongside the shape
strategy selected above -- see the selection step right after this comparison.

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import Holt


def level_naive_last(cycle_totals, cycles):
    """Extrapolate the next cycle total as simply the last observed cycle total."""
    return float(cycle_totals.sort_values("cycle_id")["cycle_total"].iloc[-1])


def level_linear_trend(cycle_totals, cycles):
    """Extrapolate one cycle total from a short linear trend (the pipeline's default)."""
    return extrapolate_cycle_total(cycle_totals.sort_values("cycle_id")["cycle_total"])


def level_holt_ets(cycle_totals, cycles):
    """Extrapolate one cycle total with Holt's linear-trend exponential smoothing (ETS)."""
    values = cycle_totals.sort_values("cycle_id")["cycle_total"].to_numpy(dtype=float)
    fit = Holt(values, initialization_method="estimated").fit()
    return max(0.0, float(fit.forecast(1)[0]))


def level_arima(cycle_totals, cycles):
    """Extrapolate one cycle total with a small ARIMA(1,1,0) model on the differenced series."""
    values = cycle_totals.sort_values("cycle_id")["cycle_total"].to_numpy(dtype=float)
    fit = ARIMA(values, order=(1, 1, 0)).fit()
    return max(0.0, float(fit.forecast(1)[0]))


def level_calendar_regression(cycle_totals, cycles):
    """Extrapolate one cycle total via OLS regression on cycle index + calendar month."""
    merged = cycle_totals.merge(cycles[["cycle_id", "open_date"]], on="cycle_id").sort_values(
        "cycle_id"
    )
    design = sm.add_constant(
        pd.DataFrame(
            {
                "cycle_id": merged["cycle_id"].astype(float),
                "month": merged["open_date"].dt.month.astype(float),
            }
        ),
        has_constant="add",
    )
    model = sm.OLS(merged["cycle_total"].astype(float), design).fit()
    next_cycle_id = merged["cycle_id"].max() + 1
    next_cycle = cycles.loc[cycles["cycle_id"].eq(next_cycle_id)].iloc[0]
    next_design = pd.DataFrame(
        {
            "const": [1.0],
            "cycle_id": [float(next_cycle_id)],
            "month": [float(next_cycle["open_date"].month)],
        }
    )
    return max(0.0, float(model.predict(next_design)[0]))


level_strategies = {
    "naive_last": level_naive_last,
    "linear_trend": level_linear_trend,
    "holt_ets": level_holt_ets,
    "arima": level_arima,
    "calendar_regression": level_calendar_regression,
}


def forecast_cycle_total_with(strategy_fn, historical_cycle_totals, cycles):
    """Apply one level-forecast strategy to every sector's historical cycle totals."""
    return (
        historical_cycle_totals.groupby("sector")
        .apply(lambda group: strategy_fn(group, cycles), include_groups=False)
        .rename("forecast_cycle_total")
        .reset_index()
    )


level_forecasts = {
    name: forecast_cycle_total_with(strategy_fn, historical_cycle_totals, cycles)
    for name, strategy_fn in level_strategies.items()
}

In [ ]:
def score_level_strategy(forecast_df, actual_cycle_total):
    """Score one level-forecast strategy's per-sector cycle totals against the actual cycle."""
    comparison = forecast_df.merge(actual_cycle_total, on="sector")
    error = comparison["forecast_cycle_total"] - comparison["actual_cycle_total"]
    mae = error.abs().mean()
    wmape = error.abs().sum() / comparison["actual_cycle_total"].sum()
    return mae, wmape


actual_cycle_total = (
    validation_orders.groupby("sector", as_index=False)["orders"]
    .sum()
    .rename(columns={"orders": "actual_cycle_total"})
)

level_scores = [
    {"strategy": name, "cycle_total_MAE": mae, "cycle_total_WMAPE": wmape}
    for name, forecast_df in level_forecasts.items()
    for mae, wmape in [score_level_strategy(forecast_df, actual_cycle_total)]
]
level_scoreboard = pd.DataFrame(level_scores).set_index("strategy").sort_values("cycle_total_WMAPE")

print("Cycle-total (level) forecast strategies scored on the held-out validation cycle")
display(level_scoreboard)

In [ ]:
def plot_level_forecasts(ax, level_forecasts, actual_cycle_total, sectors):
    """Plot each strategy's forecast cycle total for a few sectors against the actual value."""
    width = 0.8 / (len(level_forecasts) + 1)
    x = np.arange(len(sectors))
    for i, (name, forecast_df) in enumerate(level_forecasts.items()):
        values = forecast_df.set_index("sector").loc[sectors, "forecast_cycle_total"]
        ax.bar(x + i * width, values, width, label=name)
    actual_values = actual_cycle_total.set_index("sector").loc[sectors, "actual_cycle_total"]
    ax.bar(x + len(level_forecasts) * width, actual_values, width, label="actual", color="black")
    ax.set_xticks(x + len(level_forecasts) * width / 2)
    ax.set_xticklabels(sectors)
    ax.set_title("Forecast cycle total by strategy (sample sectors)")
    ax.set_ylabel("Cycle total")
    ax.legend(fontsize=8)


def plot_level_strategy_scores(ax, level_scoreboard):
    """Plot each level-forecast strategy's WMAPE against the held-out validation cycle."""
    level_scoreboard["cycle_total_WMAPE"].plot(kind="bar", ax=ax, color="tab:blue")
    ax.set_title("Strategy accuracy on the held-out cycle")
    ax.set_ylabel("Cycle-total WMAPE")
    ax.tick_params(axis="x", rotation=20)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_level_forecasts(axes[0], level_forecasts, actual_cycle_total, SECTORS[:6])
plot_level_strategy_scores(axes[1], level_scoreboard)
fig.tight_layout()
plt.show()

### Selecting the best-scoring strategies

Adopt whichever shape and level strategy scored lowest WMAPE against the held-out cycle above,
and rebuild `cycle_shape` / `forecast_cycle_totals` (and everything derived from them: the
as-is calendar preview from Part 2, and the Part 3/4 pipeline) from that choice. The plot
earlier in Part 2 was drawn before this selection and still reflects the plain-average /
linear-trend defaults; everything from here on reflects the selected strategies.

**Caveat:** this selection is made on a single held-out cycle with synthetic data, so the
score differences between strategies are small and could easily flip with a different random
seed -- treat this as a demonstration of strategy selection, not proof that the winning
strategy is truly better in general.

In [ ]:
best_shape_strategy = strategy_scoreboard["daily_WMAPE"].idxmin()
best_level_strategy = level_scoreboard["cycle_total_WMAPE"].idxmin()

cycle_shape = shape_strategies[best_shape_strategy]
forecast_cycle_totals = level_forecasts[best_level_strategy]

forecast_weekday = cycle_shape.merge(forecast_cycle_totals, on="sector")
forecast_weekday["forecast_orders"] = np.rint(
    forecast_weekday["order_share"] * forecast_weekday["forecast_cycle_total"]
).astype(int)

forecast_as_is = to_calendar(forecast_weekday, target_cycle["open_date"], current_assignment)
forecast_daily = forecast_as_is.groupby("order_date", as_index=False)["forecast_orders"].sum()

print(
    f"Selected shape strategy: {best_shape_strategy} "
    f"(daily_WMAPE={strategy_scoreboard.loc[best_shape_strategy, 'daily_WMAPE']:.3f})"
)
print(
    f"Selected level strategy: {best_level_strategy} "
    f"(cycle_total_WMAPE={level_scoreboard.loc[best_level_strategy, 'cycle_total_WMAPE']:.3f})"
)

## Part 3 -- Sector-to-slot capacitated assignment

This is the actual assignment problem: each sector is moved **as a whole** to one (block, subloco) slot -- never split across slots or days. A sector's forecasted within-window curve is reused unchanged for every candidate slot; only the slot's start day shifts where that curve lands on the calendar.

The heuristic processes sectors from largest to smallest forecast volume and, for each one, picks the feasible slot (daily load already assigned, plus this sector's shape, stays under `day_capacity` for every day) that minimizes the resulting peak daily load. If no slot is feasible, the sector keeps its current slot and is flagged. A production implementation could replace this heuristic with a MIP or CP-SAT formulation of the same generalized assignment problem while keeping the same input/output tables (see `src/solver/mip`, `src/solver/cpsat`).

In [ ]:
def compute_slot_contribution(sector, weekday_shares_by_sector, forecast_by_sector, block, subloco):
    """Per-day load a sector would add if assigned to (block, subloco), keyed by day_in_cycle."""
    start_day = slot_start_day(block, subloco)
    shares = weekday_shares_by_sector[sector]
    return {
        start_day + offset: forecast_by_sector[sector] * shares[offset]
        for offset in range(WINDOW_LENGTH)
    }


def assign_sectors_to_slots(sectors, forecast_cycle_total, weekday_shares_by_sector, day_capacity):
    """Assign each sector whole to one (block, subloco) slot, leveling peak daily load.

    Each sector keeps its own within-window curve (weekday_shares_by_sector) no matter
    which slot it lands in -- only the calendar offset (the slot's start day) changes.
    """
    day_load = {day: 0.0 for day in range(1, CYCLE_LENGTH + WINDOW_LENGTH)}
    proposal = []

    processing_order = sorted(sectors, key=lambda s: forecast_cycle_total[s], reverse=True)
    for sector in processing_order:
        current_block_, current_subloco_ = current_assignment[sector]
        best_slot = None
        best_peak = None
        for block, subloco in SLOTS:
            contribution = compute_slot_contribution(
                sector, weekday_shares_by_sector, forecast_cycle_total, block, subloco
            )
            feasible_slot = all(
                day_load[day] + load <= day_capacity for day, load in contribution.items()
            )
            if not feasible_slot:
                continue
            projected_peak = max(day_load[day] + contribution.get(day, 0.0) for day in day_load)
            if best_peak is None or projected_peak < best_peak:
                best_peak = projected_peak
                best_slot = (block, subloco)

        feasible = best_slot is not None
        chosen_block, chosen_subloco = best_slot if feasible else (current_block_, current_subloco_)
        contribution = compute_slot_contribution(
            sector, weekday_shares_by_sector, forecast_cycle_total, chosen_block, chosen_subloco
        )
        for day, load in contribution.items():
            day_load[day] += load

        proposal.append(
            {
                "sector": sector,
                "current_block": current_block_,
                "current_subloco": current_subloco_,
                "proposed_block": chosen_block,
                "proposed_subloco": chosen_subloco,
                "forecast_cycle_total": forecast_cycle_total[sector],
                "moved": (chosen_block, chosen_subloco) != (current_block_, current_subloco_),
                "feasible": feasible,
            }
        )

    return pd.DataFrame(proposal), day_load


weekday_shares_by_sector = {
    sector: cycle_shape.loc[cycle_shape["sector"].eq(sector)]
    .sort_values("weekday_offset")["order_share"]
    .to_numpy()
    for sector in SECTORS
}
forecast_cycle_total_by_sector = forecast_cycle_totals.set_index("sector")[
    "forecast_cycle_total"
].to_dict()

# Target capacity: below what the as-is (block-2-heavy) calendar would need at its peak,
# so the heuristic has to actually move sectors to level the load.
day_capacity = max(200.0, float(np.ceil(forecast_daily["forecast_orders"].max() * 0.7)))

proposed_assignment, proposed_day_load = assign_sectors_to_slots(
    SECTORS, forecast_cycle_total_by_sector, weekday_shares_by_sector, day_capacity
)

n_unfeasible = (~proposed_assignment["feasible"]).sum()
if n_unfeasible == 0:
    print("Every sector was assigned to a slot within target capacity.")
else:
    print(
        f"{n_unfeasible} sector(s) could not be moved within target capacity "
        f"and kept their current slot:"
    )
    display(proposed_assignment.loc[~proposed_assignment["feasible"]])

print(
    f"Sectors moved to a different block/subloco: "
    f"{proposed_assignment['moved'].sum()} / {len(proposed_assignment)}"
)
display(proposed_assignment.sort_values("forecast_cycle_total", ascending=False).head(15))

## Part 4 -- Validation

Two checks:

- **Forecast accuracy**: forecast requests vs the actual held-out cycle, evaluated on the as-is calendar (mean absolute error, weighted MAPE).
- **Load leveling**: as-is vs to-be daily captação -- mean, standard deviation, and peak -- plus how many sectors moved and how the per-block sector counts shifted. This mirrors the logistics metrics the real project needs to report (impact on mean, standard deviation, and peak daily captação of the CD).

In [ ]:
def compute_daily_load(assignment, forecast_cycle_total, weekday_shares_by_sector):
    """Total forecast load per day-in-cycle for a given sector -> (block, subloco) assignment."""
    day_load = {day: 0.0 for day in range(1, CYCLE_LENGTH + WINDOW_LENGTH)}
    for sector, (block, subloco) in assignment.items():
        contribution = compute_slot_contribution(
            sector, weekday_shares_by_sector, forecast_cycle_total, block, subloco
        )
        for day, load in contribution.items():
            day_load[day] += load
    return pd.Series(day_load, name="forecast_load").sort_index()


as_is_load = compute_daily_load(
    current_assignment, forecast_cycle_total_by_sector, weekday_shares_by_sector
)
proposed_map = {
    row.sector: (row.proposed_block, row.proposed_subloco)
    for row in proposed_assignment.itertuples()
}
to_be_load = compute_daily_load(
    proposed_map, forecast_cycle_total_by_sector, weekday_shares_by_sector
)

leveling_comparison = pd.DataFrame(
    {"day_in_cycle": as_is_load.index, "as_is": as_is_load.values, "to_be": to_be_load.values}
)
leveling_comparison["order_date"] = target_cycle["open_date"] + pd.to_timedelta(
    leveling_comparison["day_in_cycle"] - 1, unit="D"
)

leveling_metrics = pd.DataFrame(
    {
        "as_is": [as_is_load.mean(), as_is_load.std(), as_is_load.max()],
        "to_be": [to_be_load.mean(), to_be_load.std(), to_be_load.max()],
    },
    index=["mean_daily_load", "std_daily_load", "peak_daily_load"],
)
leveling_metrics["day_capacity"] = day_capacity

print("Daily captacao: as-is vs proposed (to-be) assignment")
display(leveling_metrics)

# Forecast accuracy against the held-out cycle, using the as-is calendar.
# actual_daily was already computed in the shape-strategy comparison above.
forecast_accuracy = forecast_daily.merge(actual_daily, how="outer", on="order_date").fillna(0)
forecast_accuracy["forecast_error"] = (
    forecast_accuracy["forecast_orders"] - forecast_accuracy["actual_orders"]
)
mae = forecast_accuracy["forecast_error"].abs().mean()
wmape = forecast_accuracy["forecast_error"].abs().sum() / forecast_accuracy["actual_orders"].sum()

forecast_metrics = pd.Series({"daily_MAE": mae, "daily_WMAPE": wmape})
print("Forecast accuracy on the held-out cycle")
display(forecast_metrics.to_frame("value"))

In [ ]:
def plot_daily_leveling(ax, leveling_comparison, day_capacity):
    """Plot as-is vs to-be daily captacao against the target capacity line."""
    ax.plot(
        leveling_comparison["order_date"], leveling_comparison["as_is"], marker="o", label="As-is"
    )
    ax.plot(
        leveling_comparison["order_date"], leveling_comparison["to_be"], marker="o", label="To-be"
    )
    ax.axhline(day_capacity, color="black", linestyle="--", label="Target capacity")
    ax.set_title("Daily captacao: as-is vs to-be")
    ax.set_ylabel("Forecast orders")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()


def plot_sectors_per_block(ax, sectors_per_block, sectors_per_block_to_be):
    """Plot the sector count per block, as-is vs to-be, as a grouped bar chart."""
    width = 0.35
    x = np.arange(len(BLOCKS))
    ax.bar(x - width / 2, sectors_per_block.values, width, label="As-is")
    ax.bar(x + width / 2, sectors_per_block_to_be.values, width, label="To-be")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Block {b}" for b in BLOCKS])
    ax.set_title("Sectors per block: as-is vs to-be")
    ax.set_ylabel("Number of sectors")
    ax.legend()


def plot_forecast_vs_actual(ax, forecast_accuracy):
    """Plot forecast vs actual daily orders for the held-out validation cycle."""
    ax.plot(
        forecast_accuracy["order_date"],
        forecast_accuracy["forecast_orders"],
        marker="o",
        label="Forecast",
    )
    ax.plot(
        forecast_accuracy["order_date"],
        forecast_accuracy["actual_orders"],
        marker="o",
        label="Actual",
    )
    ax.set_title("Requests by day: forecast vs actual (as-is calendar)")
    ax.set_ylabel("Orders")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()


def plot_leveling_metrics(ax, leveling_metrics):
    """Plot mean/std/peak daily load, as-is vs to-be, as a grouped bar chart."""
    metrics_to_plot = leveling_metrics.loc[["mean_daily_load", "std_daily_load", "peak_daily_load"]]
    metrics_to_plot[["as_is", "to_be"]].plot(kind="bar", ax=ax)
    ax.set_title("Load-leveling metrics: as-is vs to-be")
    ax.set_ylabel("Orders")
    ax.tick_params(axis="x", rotation=20)


sectors_per_block = (
    proposed_assignment.groupby("current_block")["sector"].count().reindex(BLOCKS, fill_value=0)
)
sectors_per_block_to_be = (
    proposed_assignment.groupby("proposed_block")["sector"].count().reindex(BLOCKS, fill_value=0)
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_daily_leveling(axes[0, 0], leveling_comparison, day_capacity)
plot_sectors_per_block(axes[0, 1], sectors_per_block, sectors_per_block_to_be)
plot_forecast_vs_actual(axes[1, 0], forecast_accuracy)
plot_leveling_metrics(axes[1, 1], leveling_metrics)

fig.suptitle("Validation cycle: forecast, block/subloco leveling, and capacity", fontsize=15)
fig.tight_layout()
plt.show()

## Interpretation and next steps

The leveling metrics show whether moving sectors between blocks/sublocos actually reduces the peak (and standard deviation) of daily captação relative to today's block-2-heavy as-is assignment, without exceeding target capacity on any day.

For a more realistic study, the next extensions would be: ingesting the real block/subloco calendar and per-CD/branch/fleet capacity limits (not present in `data/raw/Unifesp_Demanda.csv`, which only has historical order history), sector-specific eligibility rules (regional/logistics constraints), an exact MIP or CP-SAT formulation of this same sector-to-slot generalized assignment problem (see `src/solver/mip`, `src/solver/cpsat`, and the `docs/literature` Zotero taxonomy), and governance: proposals should apply only to future cycles, never to a cycle that's already open.